## This notebook demonstrates access to NOAA Coastal Ocean ReAnalysis (CORA) data that is stored in an S3 bucket and made freely available via NOAA's Open Data Dissemination (NODD) service. The data will be loaded as an xarray dataset. For a use case example, wave data is extracted at the nearest model node to user provided geographic coordinates and the data is displayed in a time series plot. 

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Signell, R. (2024). Reproducible Notebooks/hurricane-ike-water-levels. https://github.com/reproducible-notebooks/hurricane-ike-water-levels/tree/master***

_ _

### Input a location of interest in the format (lat, lon) ###

In [ ]:
loc = (44.61, -124.08)  # Near the Yaquina Bay north jetty, Oregon

### Import packages ###

In [ ]:
import intake
import numpy as np
import zarr

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-PAC waves (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_pac/native_grid/waves/)). ###
#### This dataset contains hourly wave data from 1979-2025 in the PAC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **timeseries** chunking here, since we are interested in a timseries at a single point.

In [ ]:
ds = catalog["CORA-PAC-swan_HS.63-timeseries"].to_dask()  # We will use Hs here, could also use Tp or Dir
ds

### Find the closest node to the location of interest (using the haversine formula), load the timeseries for the year 1979, and plot it. ###
#### The time slice of interest can be modified. The longer the time slice, the longer it will take to load the selected data into memory. ####

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    R = 6371 
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = R * c
    return km 

d = haversine(loc[0],loc[1],ds.y,ds.x)
closest_node = int(np.argmin(d.data))
ts = ds['swan_HS'].sel(node=closest_node,time=slice("1979-1-1","1979-12-31"))
ts.plot(figsize=(8,4))